# link_delay module 使用说明

这个 notebook 用来记录 `src.link_delay` 模块的常用方式。 so how


原则：

- `module/` 里的代码是通用模块，不应该因为换星座而修改。
- 星座参数、缓存路径、输出路径都通过 YAML 配置进入。
- `examples/` 里的脚本和配置是给人运行、验证、复制修改的入口。
- delay cache 通常只需要生成一次；后续多个脚本直接 import 查询模块即可。

## 1. 路径准备

如果 notebook 或脚本不是从 `E:/paper11/generic` 目录运行，需要先把 `generic` 加到 `sys.path`，这样才能 import `src.link_delay...`。

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

CONFIG_PATH = GENERIC_ROOT / "src" / "link_delay" / "examples" / "configs" / "g60_full_link_delay_store.yaml"
CONFIG_PATH

## 2. 生成 delay cache

正式推荐运行 example，而不是直接运行 `module` 文件。

G60 默认配置已经在：

`E:/paper11/generic/src/link_delay/examples/configs/g60_full_link_delay_store.yaml`

生成完整 `86164s` delay cache：

```powershell
C:\ProgramData\miniconda3\envs\paper11\python.exe E:\paper11\generic\src\link_delay\examples\build_full_link_delay_store_g60.py
```

快速验证小范围，例如 `0..10s`：

```powershell
C:\ProgramData\miniconda3\envs\paper11\python.exe E:\paper11\generic\src\link_delay\examples\build_full_link_delay_store_g60.py --start 0 --end 10
```

换星座时，不改 `module`，复制一份 YAML，然后传 `--config`。

In [ ]:
# 可选：在 notebook 里跑一个小范围验证。
# 这段默认不执行；需要时取消注释。
# import subprocess
# PYTHON_EXE = Path(r"C:\ProgramData\miniconda3\envs\paper11\python.exe")
# BUILD_EXAMPLE = GENERIC_ROOT / "src" / "link_delay" / "examples" / "build_full_link_delay_store_g60.py"
# subprocess.run([str(PYTHON_EXE), str(BUILD_EXAMPLE), "--start", "0", "--end", "10"], check=True)

## 3. 读取 YAML 配置并定位 delay store

`query_one_edge_delay.py` 只是命令行示例。真正多个文件里复用时，建议直接 import `module.query.FullLinkDelayStore`。

In [ ]:
from src.link_delay.module.build_full_link_delay_store import (
    default_delay_out_dir,
    load_yaml_dict,
    raw_config_to_dataclass,
)
from src.link_delay.module.query import FullLinkDelayStore, open_delay_store_for_interval

cfg = raw_config_to_dataclass(load_yaml_dict(CONFIG_PATH))
store_dir = cfg.paths.out_dir or default_delay_out_dir(cfg)

cfg.constellation, store_dir

## 4. 打开 delay store

`FullLinkDelayStore` 是只读查询接口，会用 mmap 读取 `edge_delay_ms.npy`，不会把整个大矩阵一次性加载到内存。

In [ ]:
store = FullLinkDelayStore(store_dir)

print("store_dir:", store.store_dir)
print("time range:", store.time_start, "..", store.time_end)
print("num_steps:", store.num_steps)
print("num_edges:", store.num_edges)

## 5. 查询某一条边在某一秒的 delay

节点编号使用全局节点编号，例如 G60 中 `node = plane * N + y`。

In [ ]:
time_step = 0
src_node = 0
dst_node = 36

delay_ms = store.delay_ms(time_step=time_step, src_node=src_node, dst_node=dst_node)
distance_km = store.distance_km(time_step=time_step, src_node=src_node, dst_node=dst_node)
edge_record = store.edge_record(src_node, dst_node)

print("edge:", edge_record)
print("delay_ms:", delay_ms)
print("distance_km:", distance_km)

## 6. 一次查询多条边

`delay_vector_for_edges` 适合某个时间片内，拿一批拓扑边的时延。后续正常拓扑是全连接图的子集时，就可以直接把子集边列表传进来。

In [ ]:
edges = [
    (0, 36),
    (0, 35),
    (0, 37),
]

delays = store.delay_vector_for_edges(time_step=0, edges=edges)
list(zip(edges, delays.tolist()))

## 7. 查询一个时间段内、一组边的 delay 矩阵

`delay_matrix_for_edges(start, end, edges)` 返回形状为 `(时间步数, 边数)` 的矩阵。

In [ ]:
delay_matrix = store.delay_matrix_for_edges(
    start=0,
    end=100,
    edges=edges,
    stride=1,
)

print(delay_matrix.shape)
delay_matrix[:3]

## 8. 按时间区间自动打开可用 store

如果你不想手动指定某个 store 目录，可以用 `open_delay_store_for_interval`。它会根据配置里的 `delay_output_base` 和星座名去找覆盖该时间区间的 delay store。

In [ ]:
store2 = open_delay_store_for_interval(
    start=0,
    end=100,
    stride=1,
    output_base=cfg.paths.delay_output_base,
    constellation_name=cfg.constellation.name,
)

store2.store_dir

## 9. 在其他 Python 文件里复用的最小模板

多个实验文件里建议复制这个模板，而不是调用 `query_one_edge_delay.py`。

In [ ]:
TEMPLATE = r'''
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.link_delay.module.build_full_link_delay_store import default_delay_out_dir, load_yaml_dict, raw_config_to_dataclass
from src.link_delay.module.query import FullLinkDelayStore

CONFIG_PATH = GENERIC_ROOT / "src" / "link_delay" / "examples" / "configs" / "g60_full_link_delay_store.yaml"
cfg = raw_config_to_dataclass(load_yaml_dict(CONFIG_PATH))
store_dir = cfg.paths.out_dir or default_delay_out_dir(cfg)
store = FullLinkDelayStore(store_dir)

delay_ms = store.delay_ms(time_step=0, src_node=0, dst_node=36)
'''

print(TEMPLATE)

## 10. 换星座时怎么做

不要修改 `src/link_delay/module`。

做法：

1. 复制 `examples/configs/g60_full_link_delay_store.yaml`。
2. 改 `constellation.name / P / N / total_sats`。
3. 改 `paths.ephem_dir / position_cache_root / delay_output_base`。
4. 运行 build example 并传入新 config。

```powershell
C:\ProgramData\miniconda3\envs\paper11\python.exe E:\paper11\generic\src\link_delay\examples\build_full_link_delay_store_from_config.py --config E:\paper11\generic\src\link_delay\examples\configs\your_constellation_full_link_delay_store.yaml
```